In [41]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import HumanMessage
from typing import Annotated
import requests
import json
import os

In [42]:
# tool create 

@tool
def get_conversion_factor(base_currency: str, target_currency: str):
    """
    This function fetches the currency conversion factor between a given base currency and target
    currency.
    """
    url = f"https://v6.exchangerate-api.com/v6/89fb63988c9e828bf60f97fb/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Given a currency conversion rate this function calculates the target currency from a given base currency value.
    """
    return base_currency_value * conversion_rate

In [43]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [44]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'NPR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1775260801,
 'time_last_update_utc': 'Sat, 04 Apr 2026 00:00:01 +0000',
 'time_next_update_unix': 1775347201,
 'time_next_update_utc': 'Sun, 05 Apr 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'NPR',
 'conversion_rate': 149.2041}

In [45]:
convert.invoke({'base_currency_value': 5, 'conversion_rate': 149.2041})

746.0205000000001

In [46]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=(
        os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
    )
)

model = ChatHuggingFace(llm=llm)

In [47]:
# tool binding
llm_with_tools = model.bind_tools([get_conversion_factor, convert])

In [48]:
messages = [HumanMessage("What is the conversion factor between USD and NPR, and based on that can you convert 30 USD to NPR")]

In [49]:
messages

[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that can you convert 30 USD to NPR', additional_kwargs={}, response_metadata={})]

In [50]:
ai_messages = llm_with_tools.invoke(messages)

In [59]:
llm_with_tools.invoke("hello")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 320, 'total_tokens': 330}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d56c3-7cca-74b3-9ea3-3671fea24eb9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 320, 'output_tokens': 10, 'total_tokens': 330})

In [51]:
ai_messages

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"NPR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_to3fl3gzddew2cdejvhvnt7h', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":30}', 'name': 'convert', 'description': None}, 'id': 'call_ayiy0fik3km4qhjh6btrwtnr', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 342, 'total_tokens': 394}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d56bd-4738-7631-9627-c68a53b74afd-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'call_to3fl3gzddew2cdejvhvnt7h', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 30}, 'id': 'call_ayiy0fik3km4qhjh6btrwtnr', 'type': 'tool_call'}], invalid_tool_calls=[], usage

In [52]:
ai_messages.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
  'id': 'call_to3fl3gzddew2cdejvhvnt7h',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 30},
  'id': 'call_ayiy0fik3km4qhjh6btrwtnr',
  'type': 'tool_call'}]

In [53]:
messages.append(ai_messages)

In [54]:
messages

[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that can you convert 30 USD to NPR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"NPR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_to3fl3gzddew2cdejvhvnt7h', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":30}', 'name': 'convert', 'description': None}, 'id': 'call_ayiy0fik3km4qhjh6btrwtnr', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 342, 'total_tokens': 394}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d56bd-4738-7631-9627-c68a53b74afd-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'call_to3fl3gzddew2cdejvhvnt7h',

In [55]:
for tool_call in ai_messages.tool_calls:
    # execute the first tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)

        # fetch this conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']

        # append this tool message to message list
        messages.append(tool_message1)

    # execute the second tool using the conversion rate from first tool
    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [56]:
messages

[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that can you convert 30 USD to NPR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"NPR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_to3fl3gzddew2cdejvhvnt7h', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":30}', 'name': 'convert', 'description': None}, 'id': 'call_ayiy0fik3km4qhjh6btrwtnr', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 342, 'total_tokens': 394}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d56bd-4738-7631-9627-c68a53b74afd-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'call_to3fl3gzddew2cdejvhvnt7h',

In [57]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and NPR is approximately 149.2041. Based on this, 30 USD is approximately equal to 4476.12 NPR.'